# iTantra Kathbath Gate A â€” data audit only
Run on **Kaggle with Internet enabled** after attaching the uploaded `kaggle_gate_a_bundle.zip` as a private Kaggle Dataset (Kaggle may extract it automatically). Add `HF_TOKEN` as a Kaggle secret. This notebook does **not** train, export, or change the app. It reads metadata for all Tamil/Telugu train shards; Kathbath audio downloads and WAVs stay under `/kaggle/working/`. Run all cells and send `itantra_gate_a_report.json` plus the displayed logs for review before any fine-tuning.

In [ ]:
import os, sys, json, zipfile, subprocess, hashlib, shutil
from pathlib import Path, PurePosixPath
assert sys.version_info >= (3, 11), 'Use a Kaggle Python 3.11+ runtime'
input_root = Path('/kaggle/input')
repo = Path('/kaggle/working/itantra')
# Kaggle usually expands uploaded ZIPs; support both input layouts.
expanded = list(input_root.rglob('bundle-provenance.json'))
bundle_paths = list(input_root.rglob('kaggle_gate_a_bundle.zip'))
if expanded:
    assert len(expanded) == 1, f'Multiple expanded iTantra bundles found: {expanded}; attach only one'
    source = expanded[0].parent
    provenance = json.loads(expanded[0].read_text(encoding='utf-8'))
    repo.mkdir(parents=True, exist_ok=True)
    for name, expected in provenance['working_files'].items():
        part = PurePosixPath(name)
        assert not part.is_absolute() and '..' not in part.parts, 'Unsafe bundle path'
        original = source/name
        assert original.is_file(), f'Missing attached bundle file: {name}'
        assert hashlib.sha256(original.read_bytes()).hexdigest() == expected, f'Bundle mismatch: {name}'
        target = repo/name
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(original, target)
    shutil.copyfile(expanded[0], repo/'bundle-provenance.json')
else:
    visible = [str(p) for p in input_root.iterdir()] if input_root.exists() else []
    assert len(bundle_paths) == 1, ('Attach one private iTantra bundle Dataset using Add Input. '
                                  f'Found {len(bundle_paths)} ZIPs and no bundle-provenance.json; visible inputs: {visible}')
    repo.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(bundle_paths[0]) as bundle:
        for name in bundle.namelist():
            part = PurePosixPath(name)
            assert not part.is_absolute() and '..' not in part.parts, 'Unsafe bundle path'
        bundle.extractall(repo)
    provenance = json.loads((repo/'bundle-provenance.json').read_text(encoding='utf-8'))
for name, expected in provenance['working_files'].items():
    part = PurePosixPath(name)
    assert not part.is_absolute() and '..' not in part.parts, 'Unsafe bundle path'
    assert hashlib.sha256((repo/name).read_bytes()).hexdigest() == expected, f'Bundle mismatch: {name}'
print('Verified attached bundle:', expanded[0] if expanded else bundle_paths[0])
os.chdir(repo)
if not os.environ.get('HF_TOKEN'):
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
assert os.environ.get('HF_TOKEN'), 'Kaggle HF_TOKEN secret missing'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub', 'pyarrow', 'datasets', 'torchcodec', 'soundfile', 'scipy', 'requests'], check=True)
print('Bundle ready; token present but never displayed. Repo:', repo)

In [ ]:
subprocess.run([sys.executable, 'tools/test_speech_leakage.py'], check=True)
subprocess.run([sys.executable, 'tools/test_kathbath_trial.py'], check=True)

## 1. Metadata-only shard and speaker plan
This reads Parquet `duration`, `speaker_id`, and `gender` columns for every shard, without opening the audio column. The output includes spread-out sample shards, cross-shard speaker overlap, selected shard IDs, and actual non-train filenames.

In [ ]:
work = Path('/kaggle/working/itantra-kathbath')
work.mkdir(parents=True, exist_ok=True)
for lang in ('ta', 'te'):
    subprocess.run([sys.executable, 'tools/kathbath_trial.py', 'plan', '--language', lang, '--target-hours', '10', '--output', str(work/f'{lang}-plan.json')], check=True)
print('Metadata plans complete; review selected shards and non-train filenames above before proceeding.')

## 2. Kathbath noisy test, held out
The publisher audio archive is about 1.4 GB. The script verifies archive sizes, inspects actual member paths, extracts only Tamil/Telugu `test` audio and transcripts, converts to 16-kHz mono WAV, and **fails** if a transcript is unmatched. Output is test-only and outside the repo. If the archive layout differs, stop and send the displayed member names; do not guess a mapping.

In [ ]:
noisy = Path('/kaggle/working/itantra-noisy')
subprocess.run([sys.executable, 'tools/kathbath_noisy_test.py', '--output-dir', str(noisy)], check=True)

## 3. Protect FLEURS + all Kathbath non-train rows + noisy test
The combined protected index contains normalized transcript hashes, stable source IDs, and 16-kHz WAV hashes. No training audio has been exported yet. A missing Kathbath `valid` split or noisy test fails closed.

In [ ]:
for lang in ('ta', 'te'):
    subprocess.run([sys.executable, 'tools/kathbath_trial.py', 'protect', '--language', lang, '--plan', str(work/f'{lang}-plan.json'), '--fleurs', f'tools/stt_training/{lang}-protected.json', '--noisy-manifest', str(noisy/f'noisy-{lang}-test-unknown.json'), '--output', f'tools/stt_training/{lang}-combined-protected.json'], check=True)

## 4. Export capped, speaker-diverse 10-hour trials
The first pass reads metadata only and filters protected transcripts/IDs; the second pass decodes only selected audio. Each speaker is capped at 20 minutes. Existing preparation code creates a speaker-disjoint dev split, then rechecks every train/dev row against the combined index. A shortage of capped hours or any overlap stops the run.

In [ ]:
for lang in ('ta', 'te'):
    subprocess.run([sys.executable, 'tools/kathbath_trial.py', 'export', '--language', lang, '--plan', str(work/f'{lang}-plan.json'), '--protected', f'tools/stt_training/{lang}-combined-protected.json', '--output-dir', str(work/lang), '--target-hours', '10', '--cap-minutes', '20'], check=True)

## 5. Leakage output and audit â€” STOP after this cell
The following calls must all print `overlaps/errors=0`. It also asserts speaker and source-ID separation, then saves a compact report. **Do not run training until the Gate A report is reviewed and approved.**

In [ ]:
report = {'status': 'Gate A only; training not run', 'languages': {}, 'bundle_provenance': provenance}
for lang in ('ta', 'te'):
    manifests = [Path(f'tools/stt_training/manifests/{lang}-{split}.jsonl') for split in ('train','dev')]
    check = subprocess.run([sys.executable, 'tools/check_speech_leakage.py', 'check', '--language', lang, '--protected', f'tools/stt_training/{lang}-combined-protected.json', '--manifest', *map(str, manifests)], check=True, capture_output=True, text=True)
    print(check.stdout.strip())
    rows = [[json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line] for path in manifests]
    assert not ({r['speaker_group'] for r in rows[0]} & {r['speaker_group'] for r in rows[1]})
    noisy_ids = {r['source_id'] for r in json.loads((noisy/f'noisy-{lang}-test-unknown.json').read_text(encoding='utf-8'))}
    assert not ({r['source_id'] for side in rows for r in side} & noisy_ids)
    report['languages'][lang] = {'plan': json.loads((work/f'{lang}-plan.json').read_text(encoding='utf-8')), 'audit': json.loads((work/lang/'prepared'/f'{lang}-audit.json').read_text(encoding='utf-8')), 'leakage_output': check.stdout.strip(), 'noisy_test_count': len(noisy_ids)}
out = Path('/kaggle/working/itantra_gate_a_report.json')
out.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
print('Gate A passed. Training NOT run. Download this report and share it:', out)
audit_lines = ['# Kathbath Gate A measured data audit', '', 'AI4Bharat Kathbath; CC-BY-4.0. README also describes CC0 packaging and IndicCorp text provenance.', '', '| Language | Train/dev hours | Train/dev clips | Train/dev speakers | Leakage |', '|---|---|---|---|---|']
for lang, result in report['languages'].items():
    a = result['audit']
    audit_lines.append(f"| {lang} | {a['hours']['train']:.3f}/{a['hours']['dev']:.3f} | {a['counts']['train']}/{a['counts']['dev']} | {a['speaker_groups_by_split']['train']}/{a['speaker_groups_by_split']['dev']} | {result['leakage_output']} |")
for lang, result in report['languages'].items():
    audit_lines.extend(['', f'## {lang} full audit and shard plan', '', '```json', json.dumps(result, ensure_ascii=False, indent=2), '```'])
audit_lines.extend(['', 'Training: Not run. Phone accuracy and two-device latency: Not verified.'])
audit_doc = repo/'docs/KATHBATH_DATA_AUDIT.md'
audit_doc.parent.mkdir(parents=True, exist_ok=True)
audit_doc.write_text('\n'.join(audit_lines)+'\n', encoding='utf-8')
print('Measured audit saved:', audit_doc)
